# uranker_utils


In [ ]:
from pathlib import Path
import sys
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AGENTS.md').is_file())
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
if str(_root / '.runtime') not in sys.path: sys.path.insert(0, str(_root / '.runtime'))
__file__ = str(_root / 'utils/uranker/uranker_utils.py')
__package__ = 'utils.uranker'
__name__ = 'utils.uranker.uranker_utils'


In [ ]:
import torch
import os
import yaml
from .uranker_model import URanker



In [ ]:
def get_option(opt_path):
    with open(opt_path, 'r') as f:
        option = yaml.safe_load(f)
    
    option.setdefault('seed', 2022)

    return option



In [ ]:
def build_model(opt):
    model_class = URanker

    # load model args
    all_args = list(opt.keys())
    model_args = {}
    for i in range(len(all_args) - 4):
        model_args[all_args[i + 4]] = opt.get(all_args[i + 4])
    model = model_class(**model_args)

    if opt['cuda']:
        model = model.cuda()
    if opt['parallel']:
        model = torch.nn.DataParallel(model)

    # load pretrained dict
    if opt['resume_ckpt_path']:
        ckpt_dict = torch.load(opt['resume_ckpt_path'], map_location='cpu', weights_only=True)['net']
        model.load_state_dict(ckpt_dict)

    return model
